# 📉 Lean Six Sigma — Phase 3: Analyze
**Goal:** Identify root causes of cycle time delays using statistical analysis and Fishbone diagram logic

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

df = pd.read_csv('../data/proposals.csv', parse_dates=['submission_date'])
before = df[df['phase'] == 'Before'].copy()
print(f'Analyzing {len(before)} baseline proposals')

In [ ]:
# Pareto Chart — Root Cause Categories
root_causes = {
    'Sequential approval loops': 38,
    'Manual data entry errors': 24,
    'Legal review backlog': 18,
    'Missing client info': 11,
    'System downtime': 6,
    'Other': 3
}

rc = pd.Series(root_causes).sort_values(ascending=False)
cumulative = rc.cumsum() / rc.sum() * 100

fig, ax1 = plt.subplots(figsize=(12, 6))
bars = ax1.bar(rc.index, rc.values, color='#378ADD', alpha=0.85, edgecolor='white')
ax1.set_ylabel('Frequency (%)', fontsize=11)
ax1.set_xlabel('Root Cause', fontsize=11)
ax1.set_title('Pareto Chart — Root Causes of Proposal Cycle Time Delays', fontsize=13, fontweight='bold')
plt.xticks(rotation=30, ha='right')

ax2 = ax1.twinx()
ax2.plot(rc.index, cumulative.values, 'o-', color='#E24B4A', linewidth=2, markersize=6, label='Cumulative %')
ax2.axhline(80, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='80% threshold')
ax2.set_ylabel('Cumulative %', fontsize=11)
ax2.set_ylim(0, 110)
ax2.legend(loc='center right')

for bar, val in zip(bars, rc.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{val}%', ha='center', fontsize=10, fontweight='500')

plt.tight_layout()
plt.savefig('../outputs/pareto_root_causes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 3 root causes account for 80% of delays — classic Pareto principle')

In [ ]:
# Hypothesis Tests — Is region/brand a significant factor?
print('=== ANOVA: Cycle Time by Region ===')
groups = [before[before['region']==r]['cycle_time_days'].values for r in before['region'].unique()]
f_stat, p_val = stats.f_oneway(*groups)
print(f'F-statistic: {f_stat:.3f} | p-value: {p_val:.4f}')
print('Conclusion:', 'Region IS a significant factor (p<0.05)' if p_val < 0.05 else 'Region is NOT a significant factor')

print('\n=== ANOVA: Cycle Time by Brand ===')
groups_b = [before[before['brand']==b]['cycle_time_days'].values for b in before['brand'].unique()]
f_stat_b, p_val_b = stats.f_oneway(*groups_b)
print(f'F-statistic: {f_stat_b:.3f} | p-value: {p_val_b:.4f}')
print('Conclusion:', 'Brand IS a significant factor (p<0.05)' if p_val_b < 0.05 else 'Brand is NOT a significant factor')

print('\n=== Rework Impact on Cycle Time ===')
rework_yes = before[before['rework_flag']==1]['cycle_time_days']
rework_no = before[before['rework_flag']==0]['cycle_time_days']
t_stat, p_t = stats.ttest_ind(rework_yes, rework_no)
print(f'With rework: {rework_yes.mean():.1f} days | Without: {rework_no.mean():.1f} days')
print(f't-statistic: {t_stat:.3f} | p-value: {p_t:.4f}')
print(f'Rework adds +{rework_yes.mean()-rework_no.mean():.1f} days on average')

In [ ]:
# Before vs After comparison
after = df[df['phase'] == 'After']['cycle_time_days']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Before vs After Improvement — Cycle Time Analysis', fontsize=13, fontweight='bold')

# Violin plot
data_plot = pd.DataFrame({'Cycle Time': pd.concat([before['cycle_time_days'], after]), 
                          'Phase': ['Before']*len(before) + ['After']*len(after)})
parts = axes[0].violinplot([before['cycle_time_days'].values, after.values], positions=[1,2], showmeans=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(['#F09595', '#5DCAA5'][i])
    pc.set_alpha(0.8)
axes[0].set_xticks([1,2])
axes[0].set_xticklabels(['Before', 'After'])
axes[0].axhline(35, color='black', linestyle='--', label='SLA 35d')
axes[0].set_title('Cycle Time Distribution')
axes[0].set_ylabel('Days')
axes[0].legend()

# Summary metrics bar chart
metrics = {'Avg Cycle Time': [before['cycle_time_days'].mean(), after.mean()],
           'SLA Breach Rate %': [before['sla_breach'].mean()*100, df[df['phase']=='After']['sla_breach'].mean()*100],
           'Rework Rate %': [before['rework_flag'].mean()*100, df[df['phase']=='After']['rework_flag'].mean()*100]}

x = np.arange(len(metrics))
w = 0.35
b1 = axes[1].bar(x - w/2, [v[0] for v in metrics.values()], w, label='Before', color='#F09595', edgecolor='white')
b2 = axes[1].bar(x + w/2, [v[1] for v in metrics.values()], w, label='After', color='#5DCAA5', edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics.keys(), rotation=15, ha='right')
axes[1].set_title('Key Metrics: Before vs After')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()